# PDA Distillation: Generalization Across Benchmarks

**Research question:** Does PDA-distilled reasoning generalize beyond GSM8K?

**Prior result (Sim 5):** GSM8K Base 24.5% -> Distilled 50.0% (+25.5pp)

**This notebook:** Train on GSM8K + MATH + ARC, evaluate Base vs Distilled on all three.

**Upload these 3 files before running:**
- `pda_training_data.jsonl` (474 GSM8K examples)
- `pda_math_training.jsonl` (200 MATH examples, 174 correct)
- `pda_arc_training.jsonl` (200 ARC examples, 186 correct)

Total training cost: $3.63 via OpenRouter. Training + eval cost: $0 (Colab GPU).

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

## 1. Load Training Data

In [ ]:
import json, re, random

def load_correct(path, domain):
    examples = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            if d.get("correct", False):
                d["domain"] = domain
                examples.append(d)
    return examples

gsm8k = load_correct("pda_training_data.jsonl", "gsm8k")
math = load_correct("pda_math_training.jsonl", "math")
arc = load_correct("pda_arc_training.jsonl", "arc")

all_correct = gsm8k + math + arc
random.seed(42)
random.shuffle(all_correct)

print(f"GSM8K:  {len(gsm8k)}")
print(f"MATH:   {len(math)}")
print(f"ARC:    {len(arc)}")
print(f"Total:  {len(all_correct)}")

## 2. Load Model + QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None, load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(model.print_trainable_parameters())

## 3. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

SYSTEM_PROMPT = """You are a problem solver who considers multiple perspectives.
1. Solve systematically, showing clear steps.
2. Look for more efficient approaches.
3. Check for edge cases and common mistakes.
4. Synthesize the best answer."""

def format_ex(ex):
    return {"conversations": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["pda_reasoning"]},
    ]}

dataset = Dataset.from_list([format_ex(ex) for ex in all_correct])
dataset = dataset.map(lambda examples: {"text": [
    tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
    for c in examples["conversations"]
]}, batched=True)

print(f"Training on {len(dataset)} examples")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, dataset_text_field="text",
    max_seq_length=max_seq_length, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=3, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear",
        seed=42, output_dir="pda-distilled-multi",
    ),
)

stats = trainer.train()
print(f"\nTraining loss: {stats.training_loss:.4f}")

## 4. Evaluation — Base vs Distilled

In [ ]:
from datasets import load_dataset

N_EVAL = 200
random.seed(42)

# --- Helpers ---
def extract_number(text):
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    if m: return float(m.group(1).replace(",", ""))
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    for n in reversed(nums):
        c = n.replace(",", "").strip()
        if c and c != "-":
            try: return float(c)
            except: continue
    return None

def extract_boxed(text):
    m = re.search(r'\\boxed\{([^}]+)\}', text)
    return m.group(1).strip() if m else None

def extract_mc(text):
    m = re.search(r'(?:answer is|Answer:?)\s*\(?([A-E])\)?\.?', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r'\(?([A-E])\)\s*$', text.strip())
    return m.group(1).upper() if m else None

def normalize(s):
    if s is None: return None
    s = str(s).strip().replace(" ", "").lower()
    if s.endswith("."): s = s[:-1]
    try: return str(float(s))
    except: return s

def generate(model, tokenizer, question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=256,
                            temperature=0.0, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

# --- Load test sets ---
print("Loading test sets...")
gsm8k_test = load_dataset("openai/gsm8k", "main", split="test")
math_test = load_dataset("EleutherAI/hendrycks_math", "algebra", split="test")
arc_test = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

def sample(ds, n):
    idx = list(range(len(ds)))
    random.shuffle(idx)
    return idx[:n]

gsm8k_idx = sample(gsm8k_test, N_EVAL)
math_idx = sample(math_test, N_EVAL)
arc_idx = sample(arc_test, N_EVAL)

# Format functions
def fmt_gsm(i): return gsm8k_test[i]["question"]
def gt_gsm(i):
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', gsm8k_test[i]["answer"])
    return float(m.group(1).replace(",", "")) if m else None
def fmt_math(i): return math_test[i]["problem"]
def gt_math(i): return extract_boxed(math_test[i]["solution"])
def fmt_arc(i):
    item = arc_test[i]
    q = item["question"]
    for l, t in zip(item["choices"]["label"], item["choices"]["text"]):
        q += f"\n({l}) {t}"
    return q
def gt_arc(i): return arc_test[i]["answerKey"]

print(f"Test sets: GSM8K {len(gsm8k_idx)}, MATH {len(math_idx)}, ARC {len(arc_idx)}")

In [ ]:
def eval_bench(model, tokenizer, name, tag, indices, fmt_fn, gt_fn, extract_fn):
    correct = total = 0
    for i, idx in enumerate(indices):
        gt = gt_fn(idx)
        if gt is None: continue
        resp = generate(model, tokenizer, fmt_fn(idx))
        pred = extract_fn(resp)
        if pred is not None and normalize(str(pred)) == normalize(str(gt)):
            correct += 1
        total += 1
        if (i+1) % 50 == 0:
            print(f"  [{tag}] {name}: {i+1}/{len(indices)} -- {correct}/{total} ({100*correct/total:.1f}%)")
    acc = 100*correct/total if total else 0
    print(f"  [{tag}] {name}: {correct}/{total} ({acc:.1f}%)")
    return {"correct": correct, "total": total, "accuracy": round(acc, 1)}

FastLanguageModel.for_inference(model)

benches = [
    ("GSM8K", gsm8k_idx, fmt_gsm, gt_gsm, extract_number),
    ("MATH",  math_idx,  fmt_math, gt_math, extract_boxed),
    ("ARC-C", arc_idx,   fmt_arc,  gt_arc,  extract_mc),
]

# === BASELINE ===
print("=" * 60)
print("BASELINE (Qwen3-1.7B without adapter)")
print("=" * 60)
model.disable_adapter_layers()
base = {}
for name, idx, fmt, gt, ext in benches:
    base[name] = eval_bench(model, tokenizer, name, "Base", idx, fmt, gt, ext)

# === DISTILLED ===
print("\n" + "=" * 60)
print("PDA-DISTILLED (Qwen3-1.7B with adapter)")
print("=" * 60)
model.enable_adapter_layers()
dist = {}
for name, idx, fmt, gt, ext in benches:
    dist[name] = eval_bench(model, tokenizer, name, "Distilled", idx, fmt, gt, ext)

## 5. Results

In [ ]:
print("\n" + "=" * 70)
print("  PDA DISTILLATION: GENERALIZATION RESULTS")
print("=" * 70)
print(f"  Student: Qwen3-1.7B | QLoRA r=16, 3 epochs")
print(f"  Training: {len(all_correct)} examples (GSM8K + MATH + ARC)")
print(f"  Cost: $3.63 data generation + $0 training")
print("=" * 70)
print(f"\n  {'Benchmark':<12} {'Base':>10} {'Distilled':>12} {'Delta':>10}")
print("  " + "-" * 44)

deltas = []
for name in ["GSM8K", "MATH", "ARC-C"]:
    b = base[name]["accuracy"]
    d = dist[name]["accuracy"]
    delta = d - b
    deltas.append(delta)
    sign = "+" if delta >= 0 else ""
    print(f"  {name:<12} {b:>8.1f}%  {d:>10.1f}%  {sign}{delta:>8.1f}pp")

print("  " + "-" * 44)
avg_b = sum(base[n]["accuracy"] for n in base) / 3
avg_d = sum(dist[n]["accuracy"] for n in dist) / 3
print(f"  {'Average':<12} {avg_b:>8.1f}%  {avg_d:>10.1f}%  +{avg_d-avg_b:>7.1f}pp")
print()

if all(d > 0 for d in deltas):
    print("  VERDICT: PDA distillation generalizes across all tested domains.")
else:
    neg = [n for n, d in zip(["GSM8K","MATH","ARC-C"], deltas) if d <= 0]
    print(f"  VERDICT: No generalization to: {', '.join(neg)}")

with open("generalization_results.json", "w") as f:
    json.dump({"base": base, "distilled": dist,
               "training_examples": len(all_correct),
               "student": "Qwen3-1.7B", "method": "QLoRA r=16, 3ep"}, f, indent=2)
print("\n  Saved to generalization_results.json")

In [ ]:
model.save_pretrained("pda-distilled-multi-lora")
tokenizer.save_pretrained("pda-distilled-multi-lora")
print("Model saved.")